# Tensor Dictionary Learning: a benchmark walkthrough

**Which tensor decomposition makes the best dictionary atom?**

Classical dictionary learning represents each signal as a sparse combination of learned
atoms. When the signals are *tensors* — image patches ($8\times8\times3$), hyperspectral
cubes ($6\times6\times16$) — you can constrain each atom to live on a low-rank tensor
manifold instead of being a free dense block. That buys parameter efficiency. The
question is what it costs in accuracy, speed, and stability.

This notebook walks through a benchmark comparing six atom parameterisations:

| Structure | Atom constraint | Knob |
|---|---|---|
| **Vectorized** | none (ordinary K-SVD) — the baseline | `n_atoms` |
| **CP** | sum of `rank` separable rank-1 terms | `rank` |
| **Tucker** | *shared* Kronecker factorisation $D = U_3 \otimes U_2 \otimes U_1$ | `mode_ranks` |
| **TT** | tensor-train cores, ranks $[1,r,r,1]$ | `rank` |
| **TR** | tensor-ring cores (wrap-around ranks) | `rank` |
| **BTD** | sum of `n_blocks` multilinear rank-$(L,L,L)$ Tucker blocks | `rank`, `n_blocks` |

### What this notebook is actually about

The headline result is not "method X wins." It's a **methodological** point that the
benchmark ran into head-first:

> Tucker originally came last. After being rebuilt to use the same optimizer and the
> same objective as everything else — changing nothing about the Tucker *structure* —
> it jumped +1.5 dB and became the sole Pareto winner.

The original ranking was measuring optimizer quality and calling it structure quality.
Section 8 reconstructs both versions so you can watch the ranking flip. Section 10 is
about the confound that *survived* the fix, because the honest answer is that the
comparison still isn't fully clean.

### Contents

1. Setup and how to run this
2. The shared harness — what makes the comparison fair
3. The six atom structures, and what each one costs
4. The datasets
5. Experiment A/B — efficiency frontier and sparsity behaviour
6. Experiment C — rank sensitivity
7. Experiment D — initialisation sensitivity
8. **The Tucker rebuild — objective mismatch, demonstrated**
9. Experiments E/F/G — real images, CP degeneracy, the BTD continuum
10. The ranking system, and the confound that remains
11. Caveats, and exercises

---
## 1. Setup

Three modules sit next to this notebook:

- `tensor_dl.py` — the models and the shared harness
- `data.py` — the three dataset builders
- `legacy_tucker.py` — a reconstruction of the *original* Tucker, used only in Section 8

`FAST = True` uses fewer seeds and fewer iterations so the whole notebook runs in a
couple of minutes. Set it to `False` to reproduce the published numbers (roughly
20–30 minutes, mostly OMP).

In [ ]:
import warnings, time, json, os
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from data import make_dataset, make_image_dataset, make_hsi_dataset
from tensor_dl import make_model, psnr, rmse

FAST = True

SEEDS      = [0, 1, 2] if FAST else [0, 1, 2, 3, 4]
N_ITER     = 6 if FAST else 10
N_TRAIN    = 200 if FAST else 320
N_TEST     = 60 if FAST else 90
ENCODE_S   = 6          # atoms allowed per sample at evaluation time
NA         = 100        # dictionary size for the K-SVD-family models

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})
COLORS = {"Vectorized": "#444444", "CP": "#d1495b", "Tucker": "#2e8b57",
          "TT": "#3b6fb6", "TR": "#e08e0b", "BTD": "#7b4fb6",
          "Tucker (legacy)": "#8fbf9f"}

print(f"FAST={FAST} | seeds={SEEDS} | n_iter={N_ITER} | train={N_TRAIN}")

---
## 2. The shared harness — what makes this a fair comparison

This is the single most important design decision in the whole project, and Section 8
is the story of what happens when you get it wrong.

Every model inherits from `BaseTensorDL` and shares **encode** and **decode**:

```python
def encode(self, X, n_nonzero):
    Y = X.reshape(X.shape[0], -1).T                 # (n_features, M)
    coef = orthogonal_mp(self.D_, Y, n_nonzero_coefs=int(n_nonzero))
    return coef.T

def decode(self, codes):
    return (codes @ self.D_.T).reshape((codes.shape[0],) + self.atom_shape)
```

Sparse coding is **OMP against a plain `(n_features, n_atoms)` matrix**, identical for
all six models. Reconstruction is a plain linear combination. So at evaluation time,
every model is doing literally the same arithmetic.

That means only two things differ between models:

1. **how the atoms are parameterised** (the constraint), and
2. **how many free parameters that costs** (`n_params`).

Which is exactly what we want to measure. Anything else that differs between models —
a different optimizer, a different training objective, a different sparsity setting —
is a **confound**: it shows up in the results looking like a property of the
decomposition, when it's really a property of your code.

Keep that sentence in mind for Section 8.

In [ ]:
import inspect
from tensor_dl import BaseTensorDL, KSVDTensorDL, TuckerDL

print(inspect.getsource(BaseTensorDL.encode))
print(inspect.getsource(BaseTensorDL.decode))

### The K-SVD engine

Vectorized / CP / TT / TR / BTD all share **one** training loop, `KSVDTensorDL.fit`.
It's textbook K-SVD with a single addition — after the rank-1 SVD atom update, the atom
is *projected back onto its tensor manifold*:

```python
E = Y[:, omega] - D @ codes[omega].T + np.outer(D[:, k], codes[omega, k])
U, S, Vt = svd(E, full_matrices=False)
atom = U[:, 0]
Tp, p, info = self._project(atom.reshape(self.atom_shape))   # <-- the only difference
```

Swap `self._project` and you swap the structure. `_proj_free` is the identity, so
"Vectorized" is just ordinary K-SVD. This is what makes the five projection-based
models genuinely comparable: same loop, same objective, one line different.

Tucker is the exception, and it has to be — its constraint is *shared across all atoms*
(one Kronecker factorisation generates the entire dictionary), so you cannot snap one
atom at a time. Section 8 is about how to handle that exception correctly.

---
## 3. The six structures, and what each one costs

Let's just look at the parameter arithmetic for an $8\times8\times3$ patch (192 numbers
if stored densely). Note the last column — it's the thing that makes Tucker strange.

In [ ]:
from tensor_dl import _proj_free, _proj_cp, _proj_tt, _proj_tr, _proj_btd

shape = (8, 8, 3)
rng = np.random.default_rng(0)
T = rng.standard_normal(shape)

rows = []
for label, fn, kw in [
    ("Vectorized (free)", _proj_free, {}),
    ("CP rank-1",         _proj_cp,   dict(rank=1)),
    ("CP rank-2",         _proj_cp,   dict(rank=2)),
    ("TT rank-2",         _proj_tt,   dict(rank=2)),
    ("TR rank-2",         _proj_tr,   dict(rank=2)),
    ("BTD L=2, 1 block",  _proj_btd,  dict(rank=2, n_blocks=1)),
]:
    Tp, nparam, info = fn(T, **kw)
    rel = np.linalg.norm(T - Tp) / np.linalg.norm(T)
    rows.append((label, nparam, 192 / nparam, rel))

print(f"{'atom structure':22} {'params/atom':>11} {'compression':>11} {'proj. error':>11}")
print("-" * 58)
for label, p, c, e in rows:
    print(f"{label:22} {p:11d} {c:10.1f}x {e:11.2f}")

Two things to read off that table:

- **Cost per atom drops fast.** A CP rank-1 atom stores 19 numbers instead of 192.
- **Projection error is large.** Squashing a *random* tensor onto a rank-1 manifold
  loses most of it. That's fine — real image patches are far from random, and K-SVD
  never asks the projector to represent noise faithfully. But it's a reminder that
  these constraints are strong.

### Tucker is a different kind of object

The five above are **per-atom** constraints: each atom independently pays its own
parameter cost, and total params $\approx$ `n_atoms` $\times$ cost-per-atom.

Tucker is a **whole-dictionary** constraint. You store three small mode factors
$U_1 \in \mathbb{R}^{8\times 9}$, $U_2 \in \mathbb{R}^{8\times 9}$,
$U_3 \in \mathbb{R}^{3\times 3}$ — and the dictionary is *every tensor product of their
columns*:

$$D = U_3 \otimes U_2 \otimes U_1$$

So the parameter count is $8{\cdot}9 + 8{\cdot}9 + 3{\cdot}3 = 153$, but the number of
atoms you get is $9 \times 9 \times 3 = 243$. **More atoms than parameters.**

That asymmetry is the source of both Tucker's spectacular efficiency score and the
confound in Section 10. Let's confirm it.

In [ ]:
mode_ranks = [9, 9, 3]
n_params = sum(d * k for d, k in zip(shape, mode_ranks))
n_atoms  = int(np.prod(mode_ranks))
print(f"Tucker mode_ranks={mode_ranks} on patch {shape}")
print(f"  parameters : {n_params}")
print(f"  atoms      : {n_atoms}")
print(f"  atoms per parameter: {n_atoms / n_params:.2f}")
print()
print(f"Compare CP rank-1 with {NA} atoms: {NA * sum(shape)} params for {NA} atoms "
      f"({NA / (NA * sum(shape)):.3f} atoms/param)")

---
## 4. The datasets

Three, in increasing order of realism:

**`make_dataset`** — synthetic, and *deliberately mixed* so no single decomposition
wins by construction. The ground-truth atoms are 40% separable rank-1 (favours CP and
Tucker), 40% low-TT-rank (favours TT and TR), 20% dense random (favours the vectorized
baseline). Each sample is a sparse combination of 3 atoms plus noise at 12 dB SNR.

**`make_image_dataset`** — real RGB patches from `skimage`'s *astronaut*, mean-removed
and unit-normalised, then noised to 12 dB. Real image structure is only *mildly*
separable, which makes this the harder and more honest test.

**`make_hsi_dataset`** — a synthetic hyperspectral cube: a few smooth spectral endmember
signatures mixed by smooth spatial abundance maps. Genuinely low multilinear-rank across
a fat 16-band spectral mode. This is the regime where tensor structure is *supposed* to
pay off and where a vectorized dictionary is starved by huge atoms.

> **Note on the source files.** As uploaded, `make_dataset` and `make_image_dataset`
> were both truncated mid-function — no `return`, so they silently handed back `None`
> and nothing downstream could run. The `data.py` shipped with this notebook has both
> tails restored (build the splits, return the dict). Nothing else was changed.

In [ ]:
ds_syn = make_dataset(atom_shape=(6, 6, 6), n_true=24, n_train=N_TRAIN, n_test=N_TEST,
                      s_true=3, snr_db=12.0, mix=(0.4, 0.4, 0.2), seed=123)
ds_img = make_image_dataset("astronaut", patch=(8, 8, 3), n_train=N_TRAIN,
                            n_test=N_TEST, snr_db=12.0, seed=7)

for nm, d in [("synthetic mixed", ds_syn), ("astronaut RGB", ds_img)]:
    print(f"{nm:16} train {d['train_noisy'].shape}  test {d['test_noisy'].shape}  "
          f"SNR {d['snr_db']} dB")

# what does a noisy patch actually look like?
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for j in range(6):
    c, n = ds_img["test_clean"][j], ds_img["test_noisy"][j]
    for i, (im, lab) in enumerate([(c, "clean"), (n, "noisy")]):
        v = (im - im.min()) / (np.ptp(im) + 1e-12)
        axes[i, j].imshow(v); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        axes[i, j].grid(False)
        if j == 0: axes[i, j].set_ylabel(lab, fontsize=9)
fig.suptitle("astronaut 8x8x3 patches at 12 dB SNR", fontsize=11)
fig.tight_layout(); plt.show()

### The evaluation protocol

Every number in this notebook is produced the same way:

1. `fit` the dictionary on **noisy training** patches (no access to clean data),
2. `encode` the **noisy test** patches with `ENCODE_S = 6` non-zero atoms,
3. `decode`, and measure PSNR against the **clean test** patches.

It's a denoising task, and the denoising happens *because* the code is sparse — a
6-atom combination can't express the noise. Nothing in the pipeline is a denoiser
except the sparsity constraint itself.

In [ ]:
def evaluate(kind, ds, seeds=None, encode_s=ENCODE_S, shape=None, **kw):
    'Fit `kind` once per seed; return dict of means/stds over seeds.'
    seeds = SEEDS if seeds is None else seeds
    shape = shape or ds["atom_shape"]
    ps, prm, ft, at, dg = [], [], [], [], []
    for sd in seeds:
        m = make_model(kind, shape, seed=sd, **kw).fit(ds["train_noisy"])
        ps.append(psnr(ds["test_clean"], m.reconstruct(ds["test_noisy"], encode_s)))
        prm.append(m.n_params); ft.append(m.fit_time_); at.append(m.n_atoms)
        if getattr(m, "degeneracy_", None) is not None:
            dg.append(m.degeneracy_)
    return dict(psnr=float(np.mean(ps)), psnr_std=float(np.std(ps)),
                psnr_all=ps, params=int(np.mean(prm)), atoms=int(np.mean(at)),
                fit_time=float(np.mean(ft)),
                degeneracy=float(np.mean(dg)) if dg else None)

print(evaluate.__doc__)

---
## 5. Experiment A/B — the efficiency frontier and sparsity behaviour

**Question A:** for a fixed operating point, where does each method land in
(model size, accuracy) space? Up-and-left is better.

**Question B:** how does accuracy respond as you loosen the sparsity budget at
*encoding* time? A dictionary that only works at high sparsity is a weak dictionary.

Both run on the **synthetic mixed** dataset, where by construction no structure should
trivially dominate.

Note the configs below: Tucker uses `mode_ranks=[7,7,7]` while everyone else uses
`n_atoms=120`. These are not matched budgets — they're each method's natural operating
point. That's a deliberate choice for the frontier plot (which has model size on its
x-axis, so unequal sizes are the *point*) and a problem for anything else. Section 10
returns to this.

In [ ]:
ATOM_SHAPE = (6, 6, 6)
NA_SYN = 120
SPARSITY_GRID = [2, 4, 6, 8, 10]
ORDER5 = ["Vectorized", "CP", "Tucker", "TT", "TR"]

def syn_specs(sd, n_iter=None):
    n_iter = n_iter or N_ITER
    return {
        "Vectorized": ("vectorized", dict(n_atoms=NA_SYN, sparsity=4, n_iter=n_iter, seed=sd)),
        "CP":         ("cp",  dict(n_atoms=NA_SYN, sparsity=4, n_iter=n_iter, rank=1, seed=sd)),
        "Tucker":     ("tucker", dict(mode_ranks=[7, 7, 7], n_iter=6, mode_sparsity=2, seed=sd)),
        "TT":         ("tt",  dict(n_atoms=NA_SYN, sparsity=4, n_iter=n_iter, rank=2, seed=sd)),
        "TR":         ("tr",  dict(n_atoms=NA_SYN, sparsity=4, n_iter=n_iter, rank=2, seed=sd)),
    }

t0 = time.time()
rec = {n: dict(params=[], atoms=[], fit_time=[], psnr=[],
               by_s={s: [] for s in SPARSITY_GRID}) for n in ORDER5}

for sd in SEEDS:
    sp = syn_specs(sd)
    for n in ORDER5:
        kind, kw = sp[n]
        m = make_model(kind, ATOM_SHAPE, **kw).fit(ds_syn["train_noisy"])
        rec[n]["params"].append(m.n_params)
        rec[n]["atoms"].append(m.n_atoms)
        rec[n]["fit_time"].append(m.fit_time_)
        rec[n]["psnr"].append(psnr(ds_syn["test_clean"],
                                   m.reconstruct(ds_syn["test_noisy"], ENCODE_S)))
        for s in SPARSITY_GRID:                       # reuse the fitted dictionary
            rec[n]["by_s"][s].append(
                psnr(ds_syn["test_clean"], m.reconstruct(ds_syn["test_noisy"], s)))
    print(f"  seed {sd} done ({time.time()-t0:.0f}s)")

print(f"\n{'model':12} {'atoms':>6} {'params':>8} {'fit(s)':>7} {'PSNR':>7} {'std':>6}")
print("-" * 50)
for n in ORDER5:
    print(f"{n:12} {int(np.mean(rec[n]['atoms'])):6d} {int(np.mean(rec[n]['params'])):8d} "
          f"{np.mean(rec[n]['fit_time']):7.2f} {np.mean(rec[n]['psnr']):7.2f} "
          f"{np.std(rec[n]['psnr']):6.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8))

for n in ORDER5:
    p, mu, sd_ = np.mean(rec[n]["params"]), np.mean(rec[n]["psnr"]), np.std(rec[n]["psnr"])
    ax1.errorbar(p, mu, yerr=sd_, fmt="o", ms=11, color=COLORS[n], capsize=4, elinewidth=1.5)
    ax1.annotate(f"  {n}", (p, mu), color=COLORS[n], fontweight="bold", fontsize=9, va="center")
ax1.set_xscale("log")
ax1.set_xlabel("dictionary parameters (log)"); ax1.set_ylabel("test PSNR (dB)")
ax1.set_title("A. Efficiency frontier\n(up-and-left is better)")

for n in ORDER5:
    mus = [np.mean(rec[n]["by_s"][s]) for s in SPARSITY_GRID]
    sds = [np.std(rec[n]["by_s"][s]) for s in SPARSITY_GRID]
    ax2.errorbar(SPARSITY_GRID, mus, yerr=sds, marker="o", color=COLORS[n],
                 label=n, capsize=3)
ax2.set_xlabel("coding sparsity (non-zero atoms per sample)")
ax2.set_ylabel("test PSNR (dB)"); ax2.set_title("B. Accuracy vs coding sparsity")
ax2.legend(frameon=False, fontsize=9)
fig.tight_layout(); plt.show()

### Reading these

**Panel A.** On synthetic data the ordering is roughly monotone in model size: the
vectorized baseline is most accurate and by far the largest; TT and TR sit close behind
at ~4x compression; CP is much smaller and pays for it; Tucker is tiny and pays a lot.
There is no free lunch visible here — which is what you'd hope from a dataset built to
mix structure types. Every point is on the frontier, so the *objective* layer of the
ranking system (Section 10) can't separate them at all on this data.

**Panel B.** Every curve rises and flattens: more atoms per sample buys accuracy with
diminishing returns. What matters is the *gap* between curves, which stays roughly
constant — the ranking isn't an artifact of the encoding budget you happened to pick.

Tucker's curve is both lowest and flattest here, and it's tempting to conclude that
$7\times7\times7$ Kronecker structure is simply too rigid for this mixed data. Hold
that thought until Section 8.

---
## 6. Experiment C — rank sensitivity

Each method has a capacity knob. Turning it up should buy accuracy and cost parameters.
The interesting question is the *exchange rate*, and whether any method has a knob that
does nothing.

Note the knobs aren't commensurable: CP/TT/TR `rank` is a per-atom rank, Tucker's knob
is the per-mode dictionary size. Panel 2 puts them on a common axis (parameters) so the
comparison means something.

In [ ]:
rank_grids = {
    "CP":     [("rank", r) for r in (1, 2, 3)],
    "TT":     [("rank", r) for r in (1, 2, 3, 4)],
    "TR":     [("rank", r) for r in (1, 2, 3)],
    "Tucker": [("mode_ranks", [r, r, r]) for r in (5, 6, 7, 8)],
}
kind_of = {"CP": "cp", "TT": "tt", "TR": "tr", "Tucker": "tucker"}
rank_res = {}

for name, grid in rank_grids.items():
    rank_res[name] = {"knob": [], "psnr": [], "std": [], "params": []}
    for key, val in grid:
        if name == "Tucker":
            kw, knob = dict(mode_ranks=val, n_iter=6, mode_sparsity=2), val[0]
        else:
            kw, knob = dict(n_atoms=NA_SYN, sparsity=4, n_iter=N_ITER, rank=val), val
        r = evaluate(kind_of[name], ds_syn, seeds=SEEDS[:2], shape=ATOM_SHAPE, **kw)
        rank_res[name]["knob"].append(knob)
        rank_res[name]["psnr"].append(r["psnr"])
        rank_res[name]["std"].append(r["psnr_std"])
        rank_res[name]["params"].append(r["params"])
    print(f"  {name:8} done")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
for name, d in rank_res.items():
    a1.errorbar(d["knob"], d["psnr"], yerr=d["std"], marker="o",
                color=COLORS[name], label=name, capsize=3)
    a2.plot(d["params"], d["psnr"], marker="o", color=COLORS[name], label=name)
a1.set_xlabel("rank knob (CP/TT/TR rank, or Tucker per-mode size)")
a1.set_ylabel("test PSNR (dB)"); a1.set_title("PSNR vs rank knob"); a1.legend(frameon=False)
a2.set_xscale("log"); a2.set_xlabel("dictionary parameters (log)")
a2.set_ylabel("test PSNR (dB)"); a2.set_title("PSNR vs model size (rank sweep)")
a2.legend(frameon=False)
fig.tight_layout(); plt.show()

TT and TR climb steeply — their rank knob is well-calibrated, buying real accuracy per
parameter. CP climbs more slowly. Tucker's curve is nearly flat: growing the per-mode
dictionaries from 5 to 8 barely moves accuracy, even though it does add parameters.

A flat capacity curve is a **diagnostic**, not just a result. If turning the knob does
nothing, either the structure genuinely can't use the capacity, or *the fitting
procedure isn't finding it*. Two candidate explanations, and this experiment can't
distinguish them.

Section 8 shows it was the second one.

---
## 7. Experiment D — initialisation sensitivity

Same config, many random seeds, look at the **spread**. This is a proxy for optimizer
quality: if the fitting procedure reliably lands in a good place, the box is short.

This is the most under-rated plot in the benchmark, because seed variance turns out to
be the signal that eventually cracks the Tucker problem open.

In [ ]:
init_seeds = [10, 11, 12, 13] if FAST else [10, 11, 12, 13, 14, 15, 16, 17]
init_res = {n: [] for n in ORDER5}
for sd in init_seeds:
    sp = syn_specs(sd)
    for n in ORDER5:
        kind, kw = sp[n]
        m = make_model(kind, ATOM_SHAPE, **kw).fit(ds_syn["train_noisy"])
        init_res[n].append(psnr(ds_syn["test_clean"],
                                m.reconstruct(ds_syn["test_noisy"], ENCODE_S)))

fig, ax = plt.subplots(figsize=(7, 4.4))
bp = ax.boxplot([init_res[n] for n in ORDER5], labels=ORDER5,
                patch_artist=True, widths=0.6, medianprops=dict(color="black"))
for patch, n in zip(bp["boxes"], ORDER5):
    patch.set_facecolor(COLORS[n]); patch.set_alpha(0.65)
ax.set_ylabel("test PSNR (dB)")
ax.set_title("D. Initialisation sensitivity across seeds\n(taller box = less robust)")
fig.tight_layout(); plt.show()

for n in ORDER5:
    a = np.array(init_res[n])
    print(f"  {n:12} {a.mean():6.2f} +- {a.std():.2f}   (min {a.min():.2f}, max {a.max():.2f})")

### The principle worth extracting

Look at *why* the boxes are the heights they are, in terms of what each fitting
procedure actually does:

| Method | Fitting subproblems | Expected spread |
|---|---|---|
| Vectorized / TT / TR | SVD truncations — each is an **optimal** linear-algebra step | tight |
| CP | ALS with a genuinely non-convex, degeneracy-prone landscape | wider |

**Seed variance is often a readout of optimizer quality, not model quality.** A method
whose subproblems are exact linear solves gives tight error bars almost regardless of
whether the model is any good. A method doing real non-convex search gives wide ones
even when the model is excellent.

Which means: if you see a method that is *both* inaccurate *and* seed-unstable, your
first hypothesis should be **the optimizer**, not the structure.

That's the hypothesis Section 8 tests.

---
# 8. The Tucker rebuild

## The centerpiece of the whole project

Across Sections 5–7, Tucker looked bad in a specific and consistent way:

- lowest accuracy on the frontier,
- flattest response to its capacity knob,
- and (on the real-image runs) the **widest** seed spread of any method.

The tempting reading is "$9\times9\times3$ separable Kronecker structure is too rigid
for real image patches." That reading would have been wrong, and the tell was the
combination — *bad **and** unstable*. Per Section 7, that pattern points at the fitting
procedure.

## What the original Tucker actually did

It learned its three mode dictionaries **completely outside the shared harness**:

1. unfold the training patches along each mode to get mode-$n$ fibers,
2. run a **separate 1-D MOD-style dictionary-learning pass** on those fibers, with its
   **own** sparsity setting, one per mode, to get $U_1, U_2, U_3$
   (subsampling fibers for speed),
3. build $D = U_3 \otimes U_2 \otimes U_1$ **once**, and stop.

Notice what is missing: at no point does the fitting procedure ever sparse-code a whole
patch against the Kronecker dictionary. The mode dictionaries were optimised to
**reconstruct raw unfolded fibers**. They were then graded on **whole-patch sparse
coding with OMP**.

**Two different objectives.** The dictionary was never told what it would be tested on.

This is worth naming precisely, because the natural guess is wrong. It was not gradient
descent. It was not a rougher version of the same alternating loop. There was no
non-convex search to speak of — MOD's update is itself a least-squares solve. The
failure mode was an **objective mismatch**, and no amount of tuning the old procedure's
hyperparameters would have fixed it.

## What the rebuild does

Put Tucker in the same loop as everything else:

```python
for _ in range(self.n_iter):
    D = _normalize_cols(self._build_Deff(U))
    codes = orthogonal_mp(D, X.reshape(M, -1).T, n_nonzero_coefs=self.sparsity).T
    Ccore = codes.reshape((M,) + tuple(ks))
    for mode in range(N):                            # separable ALS
        G  = multi_mode_dot(Ccore, [U[p] for p in other], modes=[p+1 for p in other])
        Xn = np.moveaxis(Xarr, mode+1, 0).reshape(self.atom_shape[mode], -1)
        Gn = np.moveaxis(G,   mode+1, 0).reshape(ks[mode], -1)
        U[mode] = (Xn @ Gn.T) @ inv(Gn @ Gn.T + ridge*I)
```

**OMP for coding** — the shared objective. Then **separable ALS** for the dictionary
update: solve $U_1$ holding $U_2, U_3$ fixed, then $U_2$, then $U_3$, each an exact
closed-form least-squares block. Loop.

Because Tucker's constraint is shared across the whole dictionary, this is the analogue
of the per-atom projection the other methods use: instead of snapping one atom onto its
manifold, you snap the *entire dictionary* toward Kronecker shape. Same objective, same
loop, same evaluation. **The Tucker structure is completely unchanged — 153 parameters,
243 atoms, identical `mode_ranks`.** Only the fitting changed.

Let's run both.

In [ ]:
from legacy_tucker import LegacyTuckerDL     # reconstruction of the ORIGINAL procedure

tucker_seeds = SEEDS if FAST else [0, 1, 2, 3, 4]
tk_kw = dict(mode_ranks=[9, 9, 3], n_iter=6, mode_sparsity=2)

old_ps, new_ps = [], []
for sd in tucker_seeds:
    old = LegacyTuckerDL((8, 8, 3), seed=sd, **tk_kw).fit(ds_img["train_noisy"])
    new = make_model("tucker", (8, 8, 3), seed=sd, **tk_kw).fit(ds_img["train_noisy"])
    old_ps.append(psnr(ds_img["test_clean"], old.reconstruct(ds_img["test_noisy"], ENCODE_S)))
    new_ps.append(psnr(ds_img["test_clean"], new.reconstruct(ds_img["test_noisy"], ENCODE_S)))

o, n_ = np.array(old_ps), np.array(new_ps)
print("Tucker on astronaut patches — identical structure, different fitting\n")
print(f"{'':28} {'PSNR (dB)':>14} {'params':>8} {'atoms':>7}")
print("-" * 62)
print(f"{'OLD  (per-mode MOD, outside)':28} {o.mean():7.2f} +- {o.std():4.2f} "
      f"{old.n_params:8d} {old.n_atoms:7d}")
print(f"{'NEW  (OMP + separable ALS)':28} {n_.mean():7.2f} +- {n_.std():4.2f} "
      f"{new.n_params:8d} {new.n_atoms:7d}")
print("-" * 62)
print(f"{'delta':28} {n_.mean()-o.mean():+7.2f} dB   "
      f"spread {o.std()/max(n_.std(),1e-9):4.1f}x tighter")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.4))

a1.boxplot([old_ps, new_ps], labels=["old\n(per-mode MOD)", "new\n(OMP + ALS)"],
           patch_artist=True, widths=0.5, medianprops=dict(color="black"),
           boxprops=dict(facecolor=COLORS["Tucker"], alpha=0.6))
a1.scatter([1]*len(old_ps), old_ps, color="k", zorder=3, s=18)
a1.scatter([2]*len(new_ps), new_ps, color="k", zorder=3, s=18)
a1.set_ylabel("test PSNR (dB)")
a1.set_title("Same structure, different optimizer\n(153 params either way)")

# where the two land on the frontier
a2.errorbar(old.n_params, o.mean(), yerr=o.std(), fmt="o", ms=12,
            color=COLORS["Tucker (legacy)"], capsize=4)
a2.annotate("  Tucker (old)", (old.n_params, o.mean()),
            color=COLORS["Tucker (legacy)"], fontweight="bold", fontsize=9, va="center")
a2.errorbar(new.n_params, n_.mean(), yerr=n_.std(), fmt="*", ms=20,
            color=COLORS["Tucker"], capsize=4)
a2.annotate("  Tucker (new)", (new.n_params, n_.mean()),
            color=COLORS["Tucker"], fontweight="bold", fontsize=9, va="center")
a2.annotate("", xy=(new.n_params, n_.mean()-0.08), xytext=(old.n_params, o.mean()+0.08),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="#888"))
a2.set_xscale("log"); a2.set_xlim(80, 3e4)
a2.set_xlabel("dictionary parameters (log)"); a2.set_ylabel("test PSNR (dB)")
a2.set_title("The move is straight up:\nno parameters added")
fig.tight_layout(); plt.show()

### What just happened

The published run recorded **30.66 ± 0.51 dB → 32.16 ± 0.09 dB** at an unchanged 153
parameters: about **+1.5 dB and a ~5x tighter seed spread**.

Two reasons your numbers above will differ:

- `LegacyTuckerDL` is a **reconstruction** from the description of the original, not the
  original file. At full settings it lands around 30.7 with a somewhat wider spread —
  close, but don't expect an exact match.
- **`FAST = True` shrinks the gap**, to roughly +0.6 dB. Fewer training patches hurt the
  rebuilt Tucker more than the old one, because ALS against the shared objective is the
  part that actually benefits from data — the old procedure was fitting fibers, and
  there are plenty of those either way. Set `FAST = False` to see the full effect.

The direction and the *joint* improvement in both mean and spread are robust either way,
and those are what the argument rests on.

Both things improved **together**, and that's the signature of an optimizer fix rather
than a modelling change:

- accuracy went up because the dictionary is now optimised for the objective it's graded on;
- variance went down because every subproblem is now an **exact linear solve** (OMP
  selection aside), so there's very little for the seed to perturb.

A structural change would typically trade these off. Fixing a mismatch improves both.

### The consequence for every earlier plot

This invalidates the Tucker point in Sections 5, 6 and 7 — and, importantly, **not the
others**, since only Tucker was outside the harness. Concretely:

- The flat rank-sensitivity curve in Section 6 was the old procedure failing to convert
  capacity into accuracy, not Tucker being unable to use it.
- The wide seed spread was a fitting artifact.
- The frontier position was wrong by 1.5 dB in the direction that made the whole
  headline conclusion ("CP wins, Tucker mediocre") backwards.

### The general lesson

> A benchmark can only compare what it holds constant. Anything that varies between
> arms *other than the thing under study* gets silently attributed to the thing under
> study.

Here the un-held-constant variable was the training objective, and it cost 1.5 dB —
larger than every genuine structural difference in the entire benchmark. The result
that "CP beats Tucker" was, substantially, the result that "our CP code was written
better than our Tucker code."

This is not a Tucker-specific hazard. Any time one arm of a comparison has a bespoke
implementation path, assume this until you've ruled it out.

---
## 9. Experiments E/F/G — real images, CP degeneracy, the BTD continuum

Now with Tucker on equal footing, the real-image benchmark. BTD joins here as a sixth
method: it interpolates between CP ($L=1$) and Tucker (one full-rank block), so it's a
useful probe of whether the CP-vs-Tucker gap is a smooth continuum or a cliff.

In [ ]:
SHAPE_IMG = (8, 8, 3)
ORDER6 = ["Vectorized", "CP", "Tucker", "TT", "TR", "BTD"]
img_specs = {
    "Vectorized": ("vectorized", dict(n_atoms=NA, sparsity=4, n_iter=N_ITER)),
    "CP":         ("cp",     dict(n_atoms=NA, sparsity=4, n_iter=N_ITER, rank=1)),
    "Tucker":     ("tucker", dict(mode_ranks=[9, 9, 3], n_iter=6, mode_sparsity=2)),
    "TT":         ("tt",     dict(n_atoms=NA, sparsity=4, n_iter=N_ITER, rank=2)),
    "TR":         ("tr",     dict(n_atoms=NA, sparsity=4, n_iter=N_ITER, rank=2)),
    "BTD":        ("btd",    dict(n_atoms=NA, sparsity=4, n_iter=N_ITER, rank=2,
                                  proj_kw=dict(n_blocks=1))),
}

t0 = time.time()
real = {}
for n in ORDER6:
    kind, kw = img_specs[n]
    real[n] = evaluate(kind, ds_img, shape=SHAPE_IMG, **kw)
    r = real[n]
    print(f"  {n:12} PSNR {r['psnr']:5.2f}+-{r['psnr_std']:.2f}  "
          f"params {r['params']:6d}  atoms {r['atoms']:4d}  fit {r['fit_time']:5.2f}s")
print(f"\ntotal {time.time()-t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5))
for n in ORDER6:
    r = real[n]
    ax.errorbar(r["params"], r["psnr"], yerr=r["psnr_std"], fmt="o", ms=12,
                color=COLORS[n], capsize=4)
    ax.annotate("  " + n, (r["params"], r["psnr"]), color=COLORS[n],
                fontweight="bold", fontsize=9, va="center")
ax.set_xscale("log")
ax.set_xlabel("dictionary parameters (log)"); ax.set_ylabel("test PSNR (dB)")
ax.set_title("E. Real RGB patches (astronaut 8x8x3)\nwith Tucker in the shared harness")
fig.tight_layout(); plt.show()

On real image patches the picture is completely different from the synthetic one:
Tucker now sits alone in the top-left corner — most accurate **and** smallest by an
order of magnitude. Everything else is bunched together around 31.3 dB at 2–20k
parameters.

Why real images and not the synthetic mix? Because natural image patches genuinely are
close to separable — horizontal structure, vertical structure, and colour are largely
independent — which is exactly the prior that a Kronecker dictionary encodes. This is
the standard argument for separable dictionaries in image processing, and here it shows
up as a 100x parameter saving at *better* accuracy.

The synthetic dataset, by contrast, was built to be 40% low-TT-rank and 20% dense —
structure Kronecker cannot express. Both results are correct; they're about different data.

### F. CP degeneracy — a case study in reading error bars

CP's ALS is the one genuinely non-convex fit in the benchmark, and it has a
characteristic failure: **degeneracy**, where components diverge in magnitude while
nearly cancelling. The diagnostic:

$$\text{deg} = \frac{\sum_r \prod_n \|f_n^{(r)}\|}{\|\hat{T}\|}$$

Around 1 means well-separated components. Much greater than 1 means large components
that mostly annihilate each other — numerically fragile and a sign the fit is falling
apart.

In [ ]:
cp_ranks = (1, 2, 3)
cp_res = {ci: {"rank": [], "psnr": [], "std": [], "deg": []} for ci in ("svd", "random")}
for ci in ("svd", "random"):
    for r in cp_ranks:
        out = evaluate("cp", ds_img, shape=SHAPE_IMG, n_atoms=80, sparsity=4,
                       n_iter=6, rank=r, init=("data" if ci == "svd" else "random"),
                       proj_kw=dict(cp_init=ci))
        cp_res[ci]["rank"].append(r); cp_res[ci]["psnr"].append(out["psnr"])
        cp_res[ci]["std"].append(out["psnr_std"]); cp_res[ci]["deg"].append(out["degeneracy"])
        print(f"  {ci:6} rank {r}: PSNR {out['psnr']:5.2f}+-{out['psnr_std']:.2f}  "
              f"deg {out['degeneracy']:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
for ci, ls, mk in (("svd", "-", "o"), ("random", "--", "s")):
    a1.errorbar(cp_res[ci]["rank"], cp_res[ci]["psnr"], yerr=cp_res[ci]["std"],
                ls=ls, marker=mk, color=COLORS["CP"], capsize=3, label=f"CP ({ci} init)")
    a2.plot(cp_res[ci]["rank"], cp_res[ci]["deg"], ls=ls, marker=mk,
            color="#444", label=f"{ci} init")
a1.set_xlabel("CP atom rank"); a1.set_ylabel("test PSNR (dB)")
a1.set_xticks(list(cp_ranks)); a1.set_title("CP accuracy & seed-spread vs rank")
a1.legend(frameon=False)
a2.axhline(1.0, color="green", lw=1, ls=":", label="no degeneracy")
a2.set_xlabel("CP atom rank"); a2.set_ylabel("degeneracy ratio")
a2.set_xticks(list(cp_ranks)); a2.set_title("CP degeneracy diagnostic")
a2.legend(frameon=False)
fig.tight_layout(); plt.show()

Two findings here, and the second is a nice trap.

**Accuracy *falls* as CP rank rises.** More capacity per atom makes things worse. That's
degeneracy: the right panel shows the diagnostic climbing steeply above 1 with rank, and
faster under random init. The extra components aren't representing extra structure,
they're diverging and cancelling.

**Random init beat SVD init.** Counter-intuitive, and a good reminder that "smarter
initialisation" is a hypothesis, not a guarantee — an SVD start can drop you into a
basin that ALS then has trouble leaving.

Either way, CP at rank 1 is the sensible operating point, which is what the main
benchmark uses.

### G. The BTD continuum

BTD sums `n_blocks` multilinear rank-$(L,L,L)$ Tucker blocks. $L=1$ recovers CP; one
full-rank block recovers Tucker. Sweeping $L$ and `n_blocks` traces the path between them.

In [ ]:
btd_grid = [("L1xb1", 1, 1), ("L1xb2", 1, 2), ("L2xb1", 2, 1), ("L2xb2", 2, 2)]
if not FAST:
    btd_grid += [("L1xb3", 1, 3), ("L3xb1", 3, 1)]

btd_pts = {"label": [], "params": [], "psnr": [], "std": []}
for label, L, nb in btd_grid:
    out = evaluate("btd", ds_img, shape=SHAPE_IMG, n_atoms=NA, sparsity=4,
                   n_iter=max(6, N_ITER-2), rank=L, proj_kw=dict(n_blocks=nb))
    btd_pts["label"].append(label); btd_pts["params"].append(out["params"])
    btd_pts["psnr"].append(out["psnr"]); btd_pts["std"].append(out["psnr_std"])
    print(f"  BTD {label}: params={out['params']:6d}  PSNR={out['psnr']:.2f}")

cp_ref = {"params": [], "psnr": []}
for r in (1, 2, 3):
    out = evaluate("cp", ds_img, shape=SHAPE_IMG, n_atoms=NA, sparsity=4,
                   n_iter=max(6, N_ITER-2), rank=r, seeds=SEEDS[:2])
    cp_ref["params"].append(out["params"]); cp_ref["psnr"].append(out["psnr"])

fig, ax = plt.subplots(figsize=(7.6, 5))
o = np.argsort(btd_pts["params"])
ax.errorbar(np.array(btd_pts["params"])[o], np.array(btd_pts["psnr"])[o],
            yerr=np.array(btd_pts["std"])[o], fmt="o-", color=COLORS["BTD"],
            capsize=3, ms=9, label="BTD (vary L, #blocks)")
for lab, x, y in zip(btd_pts["label"], btd_pts["params"], btd_pts["psnr"]):
    ax.annotate(lab, (x, y), fontsize=8, color=COLORS["BTD"],
                xytext=(0, 8), textcoords="offset points", ha="center")
ax.plot(cp_ref["params"], cp_ref["psnr"], "s--", color=COLORS["CP"], ms=9,
        label="CP (rank 1-3)")
ax.plot([real["Tucker"]["params"]], [real["Tucker"]["psnr"]], "D",
        color=COLORS["Tucker"], ms=13, label="Tucker")
ax.set_xscale("log")
ax.set_xlabel("dictionary parameters (log)"); ax.set_ylabel("test PSNR (dB)")
ax.set_title("G. BTD traces the CP<->Tucker continuum\n(L = block multilinear rank)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); plt.show()

BTD's sweep overlaps CP's almost exactly and slopes the same way — *down* as you add
parameters, for the same degeneracy-flavoured reason. And critically, **BTD never gets
anywhere near Tucker**, which sits far off to the left.

This is worth dwelling on, because BTD's whole selling point is that it interpolates
between CP and Tucker — so why doesn't it reach Tucker?

Because BTD is a **per-atom** decomposition: 100 independently-parameterised atoms, each
built from its own Tucker blocks. Tucker here is a **shared** factorisation: three small
mode dictionaries generating all 243 atoms. Those are different animals. The continuum
BTD traces is real, but it's a continuum of per-atom structures, and the Tucker point on
this chart isn't on it. Tucker's advantage comes from *amortising* structure across the
whole dictionary, which no per-atom method can imitate.

---
## 10. The ranking system — and the confound that survived

"Which is most efficient?" has no scalar answer, because efficiency is at least
four-dimensional: **accuracy**, **parameter efficiency**, **speed**, **robustness**.
Any single number smuggles in a weighting, and the weighting is a value judgement about
your deployment, not a fact about the methods.

So `ranking.py` uses three layers, ordered from objective to opinionated:

**1. Pareto layer (weight-free).** A method is *dominated* if another is at least as
good on every axis and strictly better on one. Dominated methods can't win under **any**
non-negative weighting, so they're out of contention — no opinion required. In practice
this is computed on the two core axes (accuracy, parameter efficiency); a 4-axis Pareto
is non-discriminating here because nearly every method wins *something*.

**2. Interpretable scalar.** $E = \text{PSNR} / \log_{10}(\text{params})$ — "accuracy
retained per decade of parameters." Unit-ful, needs no normalisation, easy to argue with.

**3. Profile-weighted composite.** Min-max each axis to $[0,1]$, weighted-sum under a
named profile (Balanced / Quality / Compression / Realtime), grade S–D. Fully
opinionated, but the opinion is written down as explicit weights you can change.

In [ ]:
PROFILES = {   # weights over (accuracy, param_eff, speed, robustness)
    "Balanced":    (0.40, 0.30, 0.15, 0.15),
    "Quality":     (0.60, 0.20, 0.10, 0.10),
    "Compression": (0.25, 0.55, 0.10, 0.10),
    "Realtime":    (0.30, 0.20, 0.40, 0.10),
}
TIERS = [("S", 0.80), ("A", 0.65), ("B", 0.45), ("C", 0.25), ("D", 0.0)]

def _minmax(vals):
    lo, hi = min(vals), max(vals)
    return [0.5]*len(vals) if hi-lo < 1e-12 else [(v-lo)/(hi-lo) for v in vals]

def build_ranking(raw):
    names = list(raw)
    A = _minmax([raw[n]["psnr"] for n in names])
    P = _minmax([-np.log10(raw[n]["params"]) for n in names])
    S = _minmax([-np.log10(raw[n]["fit_time"]) for n in names])
    R = _minmax([-raw[n]["psnr_std"] for n in names])
    ax = {n: dict(accuracy=A[i], param_eff=P[i], speed=S[i], robustness=R[i])
          for i, n in enumerate(names)}
    E = {n: raw[n]["psnr"]/np.log10(raw[n]["params"]) for n in names}

    def dominates(b, a):   # does b dominate a on (accuracy, param_eff)?
        ge = (ax[b]["accuracy"] >= ax[a]["accuracy"]-1e-9 and
              ax[b]["param_eff"] >= ax[a]["param_eff"]-1e-9)
        gt = (ax[b]["accuracy"] >  ax[a]["accuracy"]+1e-9 or
              ax[b]["param_eff"] >  ax[a]["param_eff"]+1e-9)
        return ge and gt
    pareto = {n: not any(dominates(o, n) for o in names if o != n) for n in names}

    comp = {n: {p: sum(w*ax[n][k] for w, k in
                       zip(ws, ("accuracy", "param_eff", "speed", "robustness")))
                for p, ws in PROFILES.items()} for n in names}
    tier = lambda s: next(t for t, c in TIERS if s >= c)
    rows = [dict(name=n, **raw[n], axes=ax[n], E=round(E[n], 2), pareto=pareto[n],
                 composites=comp[n], tier=tier(comp[n]["Balanced"])) for n in names]
    rows.sort(key=lambda r: r["composites"]["Balanced"], reverse=True)
    return rows

rows = build_ranking(real)
print(f"{'#':>2} {'method':11} {'tier':>4} {'PSNR':>6} {'params':>7} "
      f"{'E':>6} {'Pareto':>7} {'balanced':>9}")
print("-" * 60)
for i, r in enumerate(rows, 1):
    print(f"{i:>2} {r['name']:11} {r['tier']:>4} {r['psnr']:6.2f} {r['params']:7d} "
          f"{r['E']:6.2f} {'FRONT' if r['pareto'] else 'dom':>7} "
          f"{r['composites']['Balanced']:9.3f}")

print("\nWinner by profile:")
for p in PROFILES:
    order = sorted(rows, key=lambda r: r["composites"][p], reverse=True)
    print(f"  {p:11}: {order[0]['name']:11} ({' > '.join(r['name'] for r in order)})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for r in rows:
    ax1.scatter(r["params"], r["psnr"], marker="*" if r["pareto"] else "o",
                s=360 if r["pareto"] else 150, color=COLORS[r["name"]],
                edgecolor="black", linewidth=1.2 if r["pareto"] else 0.5, zorder=3)
    ax1.annotate("  " + r["name"], (r["params"], r["psnr"]), color=COLORS[r["name"]],
                 fontweight="bold", fontsize=9, va="center")
front = sorted([r for r in rows if r["pareto"]], key=lambda r: r["params"])
ax1.plot([r["params"] for r in front], [r["psnr"] for r in front],
         "k--", lw=1, alpha=0.6, label="Pareto frontier")
ax1.set_xscale("log"); ax1.set_xlabel("dictionary parameters (log)")
ax1.set_ylabel("test PSNR (dB)")
ax1.set_title("Objective layer: Pareto frontier\n(stars = non-dominated)")
ax1.legend(frameon=False, loc="lower right")

profs = list(PROFILES)
M = np.array([[r["composites"][p] for p in profs] for r in rows])
im = ax2.imshow(M, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax2.set_xticks(range(len(profs))); ax2.set_xticklabels(profs, rotation=20, ha="right")
ax2.set_yticks(range(len(rows))); ax2.set_yticklabels([r["name"] for r in rows])
ax2.grid(False)
for i in range(len(rows)):
    for j in range(len(profs)):
        ax2.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=9,
                 color="white" if M[i, j] < 0.6 else "black")
ax2.set_title("Opinionated layer: composite by use-case profile")
fig.colorbar(im, ax=ax2, fraction=0.046, label="composite (0-1)")
fig.tight_layout(); plt.show()

### The confound that survived the fix

Tucker sweeps every profile and is the sole Pareto winner. Before quoting that anywhere,
read the next paragraph.

Fixing the objective mismatch removed one confound and **exposed the next one**:

> Tucker carries **243 atoms** (its $9\times9\times3$ Kronecker grid) versus **100** for
> every other method.

At a fixed coding sparsity of 6, having 243 atoms to choose from is an advantage
independent of whether Kronecker structure is a good prior. So part of Tucker's accuracy
edge is *more atoms*, not *better structure*.

Is that cheating? It depends entirely on which axis you're scoring:

- On **parameter efficiency**, it's completely legitimate. Tucker really does deliver 243
  usable atoms for 153 stored numbers. If your constraint is storage, this is a real win
  and the ranking is telling the truth.
- On **structural quality**, it is not a clean comparison. The arms differ in atom count,
  so the experiment cannot separate "Kronecker is a good prior" from "more atoms help."

The defensible claim is therefore narrow and specific:

> With the same optimizer and the same objective, the separable Kronecker structure is
> extraordinarily **parameter-efficient** on real image patches.

Not "Tucker is best." And there's direct evidence for the caution: on the hyperspectral
cube at **equal parameter budget**, Tucker no longer dominates.

Note the shape of this. Section 8 was a confound that made a method look *worse* than it
was; Section 10 is a confound making the same method look *better* than it is. Removing
one revealed the other. This is normal — confounds are layered, and "we fixed the
confound" almost always means "we fixed the confound we could see."

The clean next step is a **matched-budget harness**: hold `n_atoms` equal across methods
and let parameters vary, as a companion to the current matched-`n_atoms`-free setup.
Section 11 has it as an exercise.

In [ ]:
# The published 5-seed ranking, for reference.
#
# With FAST=True every method comes in ~0.5-1.0 dB low, because 200 training patches
# and 6 K-SVD iterations is a weaker fit than 320 and 10. The offset is roughly uniform,
# so the ORDERING and the Pareto structure survive -- which is the thing to check here.
# Set FAST=False to reproduce the absolute numbers.
published = {
    "Tucker":     dict(psnr=32.16, psnr_std=0.09, params=153,   E=14.72, tier="S", pareto=True),
    "CP":         dict(psnr=31.71, psnr_std=0.11, params=1900,  E=9.67,  tier="C", pareto=False),
    "BTD":        dict(psnr=31.38, psnr_std=0.06, params=4600,  E=8.57,  tier="C", pareto=False),
    "TR":         dict(psnr=31.31, psnr_std=0.09, params=7600,  E=8.07,  tier="C", pareto=False),
    "TT":         dict(psnr=31.28, psnr_std=0.07, params=5400,  E=8.38,  tier="C", pareto=False),
    "Vectorized": dict(psnr=30.80, psnr_std=0.05, params=19200, E=7.19,  tier="D", pareto=False),
}
print(f"{'method':12} {'published':>18} {'this run':>18}   {'delta':>7}")
print("-" * 62)
for n, p in published.items():
    r = real[n]
    print(f"{n:12} {p['psnr']:8.2f}+-{p['psnr_std']:.2f}      "
          f"{r['psnr']:8.2f}+-{r['psnr_std']:.2f}   {r['psnr']-p['psnr']:+7.2f}")

print("\nNote: ranker.html as shipped still hardcodes the OLD Tucker numbers")
print("(30.66 dB, E=14.04) and a verdict line claiming CP and Tucker are jointly")
print("Pareto-optimal. It predates the rebuild and needs regenerating from ranking.json.")

---
## 11. Caveats, and what to do next

### What this benchmark does and does not establish

It ranks **these implementations** on **this benchmark** — one image, one noise level,
one patch size, one operating point per method. The reusable artifact is the
*methodology*: the shared harness, the confound discipline, the three-layer ranking.
The numbers should be re-derived on your own data before being quoted as properties of
the decompositions themselves.

Specific limitations to state up front:

1. **Atom count is not matched** (Section 10) — Tucker's 243 vs everyone's 100.
2. **One image.** *astronaut* is a single natural image. Separability is a property of
   the data, and this is $n=1$.
3. **One noise level** (12 dB) and **one encode sparsity** (6). Section 5B shows the
   ordering is stable across sparsity, but that's one axis of one dataset.
4. **Fit-time comparisons are soft.** The `speed` axis mixes algorithmic cost with
   implementation quality. Tucker being fastest is partly that it optimises 153 numbers
   rather than 100 atoms — real, but partly an artifact of Python-level loop counts.
5. **`ranker.html` is stale** — it still shows pre-rebuild Tucker numbers.

### The through-line worth keeping

If you take one thing from this notebook, take the diagnostic from Sections 7 and 8:

> Fitting procedures whose subproblems are **exact linear solves** (ALS blocks, SVD
> truncations, MOD updates) give tight error bars. Genuinely **non-convex search** gives
> wide ones. So **seed variance is often a readout of optimizer quality, not model
> quality** — and a method that is both inaccurate *and* unstable should make you
> suspect your code before you conclude anything about the mathematics.

That's what CP's wide early error bars (non-convex ALS, measurable degeneracy) versus
TT's tight ones (a sequence of optimal truncations) were telling us all along, and it's
what cracked the Tucker problem open.

### Exercises

**1. Matched-atom-budget harness (the important one).** Give every method 243 atoms, or
give Tucker `mode_ranks` yielding ~100. Does Tucker's accuracy edge survive? This is the
experiment that would settle structure-vs-atom-count.

**2. Hyperspectral.** `make_hsi_dataset` builds a $6\times6\times16$ cube with genuine
low multilinear rank across a fat spectral mode, and it has a **validation split** the
RGB set lacks. The claim to check is that at equal parameter budget Tucker no longer
dominates there.

**3. Break something on purpose.** Give one method a different `sparsity` during `fit`
than the others and re-run Section 5. How large a spurious gap can you manufacture?
Compare it to the real structural differences.

**4. Re-weight the ranking.** The composite layer is honest because the weights are
explicit. Find a defensible weighting under which Tucker is *not* first, and one under
which the vectorized baseline is.

In [ ]:
# Exercise 2 starter: the hyperspectral cube, with a validation split.
ds_hsi = make_hsi_dataset(patch=(6, 6, 16), n_endmembers=5, img=(72, 72),
                          n_train=150, n_val=50, n_test=50, snr_db=12.0, seed=0)
print("HSI splits:", {k: v.shape for k, v in ds_hsi.items() if hasattr(v, "shape")})
print("\nA dense atom here is 6*6*16 =", 6*6*16, "numbers -- the vectorized")
print("baseline is starved, and tensor structure should pay off. Over to you.")

---

*Built from `tensor_dl.py`, `data.py`, `ranking.py`, `run_benchmark.py`, `run_extended.py`
and `ranking.json`. `legacy_tucker.py` is a reconstruction of the original Tucker
procedure from its description, included so Section 8 is executable rather than merely
asserted; it reproduces the old result closely but not exactly.*